# Gemma 4 Failure Analysis — Robot Rollout Diagnosis

Preliminary experiment: can Gemma 4 (26B-A4B) understand robot rollout videos
well enough to diagnose failures, rate progress, and suggest corrections?

**Gating question**: If Gemma 4 can't reliably do this, the VLM-as-reward-judge direction is dead.
If it can, we have strong motivation for the self-improvement loop.

In [ ]:
import json
import re
from pathlib import Path

import av
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video, display
from PIL import Image
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_ID = "google/gemma-4-26B-A4B-it"

## 1. Load Gemma 4 26B-A4B

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
)
print("Model loaded on", model.device)

## 2. Pick a rollout video

Change `VIDEO_PATH` to analyze a different rollout.

In [ ]:
EVAL_DIR = Path(
    "checkpoints/gr00t_n1-5/multitask_learning/checkpoint-120000/evals"
)

failure_videos = sorted(EVAL_DIR.rglob("failure_*.mp4"))
success_videos = sorted(EVAL_DIR.rglob("success_*.mp4"))
print(f"Found {len(failure_videos)} failure videos, {len(success_videos)} success videos")

for v in failure_videos:
    print(f"  {v.parent.name}/{v.name}")

In [ ]:
# Pick a video to analyze
VIDEO_PATH = failure_videos[0]  # change index to pick a different one
print(f"Analyzing: {VIDEO_PATH.name}")

# Display the video inline
display(Video(str(VIDEO_PATH), embed=True, width=768))

## 3. Visualize sampled frames

See what Gemma 4 will "see" when using frame-based input.

In [ ]:
def sample_frames(video_path: str, n_frames: int = 16) -> list[Image.Image]:
    container = av.open(str(video_path))
    total = container.streams.video[0].frames or 250
    targets = set(np.linspace(0, total - 1, n_frames, dtype=int).tolist())
    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i in targets:
            frames.append(frame.to_image())
        if i > max(targets):
            break
    container.close()
    return frames


def extract_task_description(video_path: Path) -> str:
    match = re.match(r"(?:success|failure)_(\w+?)_(.*?)_try_\d+", video_path.stem)
    if match:
        return f"{match.group(1)}: {match.group(2).replace('_', ' ')}"
    return video_path.stem.replace("_", " ")


def generate(messages: list[dict], max_new_tokens: int = 1024) -> str:
    inputs = processor.apply_chat_template(
        messages, tokenize=True, return_dict=True, return_tensors="pt", add_generation_prompt=True,
    ).to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.decode(output[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)


frames = sample_frames(VIDEO_PATH, n_frames=8)
task_desc = extract_task_description(VIDEO_PATH)
print(f"Task: {task_desc}")
print(f"Sampled {len(frames)} frames, resolution: {frames[0].size}")

fig, axes = plt.subplots(2, 4, figsize=(20, 6))
for i, (ax, frame) in enumerate(zip(axes.flat, frames)):
    ax.imshow(frame)
    ax.set_title(f"Frame {i}")
    ax.axis("off")
plt.suptitle(f"Task: {task_desc}", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Run Gemma 4 analysis

Sample frames with PyAV, send as images to Gemma 4.

In [ ]:
ANALYSIS_PROMPT = """\
You are analyzing a robot manipulation rollout in a kitchen simulation (RoboCasa).
The video shows 3 camera views side-by-side: left view, right view, and wrist camera (eye-in-hand).

**Task instruction**: {task_description}

Analyze this rollout and answer:

1. **Success or failure?** Did the robot complete the task?
2. **Progress score** (0.0 to 1.0): How far did the robot get toward completing the task?
3. **Failure point**: At what moment in the video did things go wrong? Describe the frame/timepoint.
4. **Root cause**: What specifically went wrong? Choose from:
   - Object distraction (attended to wrong object)
   - Grasp failure (missed the grasp)
   - Wrong interaction point (e.g., pressed wrong button)
   - No retry after failure (repeated the same failing strategy)
   - Spatial error (wrong position/angle of approach)
   - Other (describe)
5. **Corrective suggestion**: What should the robot do differently on the next attempt?
6. **Dense progress timeline**: Break the video into ~5 phases and rate progress at each phase (0.0-1.0).

Respond in JSON format:
{{
    "success": false,
    "progress_score": 0.3,
    "failure_point": "Around 5 seconds in, when the robot...",
    "root_cause": "wrong_interaction_point",
    "root_cause_detail": "The robot pressed the left button instead of the right one",
    "corrective_suggestion": "Approach the panel from a different angle and target the right button",
    "progress_timeline": [
        {{"phase": "0-2s", "description": "Robot navigates toward the target object", "score": 0.2}},
        {{"phase": "2-4s", "description": "Robot reaches for the object", "score": 0.4}}
    ]
}}
"""

In [ ]:
def analyze_video(video_path: Path, n_frames: int = 16) -> dict:
    task_desc = extract_task_description(video_path)
    frames = sample_frames(str(video_path), n_frames)
    prompt = ANALYSIS_PROMPT.format(task_description=task_desc)
    messages = [{
        "role": "user",
        "content": [{"type": "image", "image": f} for f in frames] + [{"type": "text", "text": prompt}],
    }]
    raw = generate(messages)
    json_match = re.search(r"\{.*\}", raw, re.DOTALL)
    result = json.loads(json_match.group()) if json_match else {"raw_response": raw}
    result["video_path"] = str(video_path)
    result["task_description"] = task_desc
    return result

In [ ]:
frames = sample_frames(VIDEO_PATH, n_frames=16)
prompt = ANALYSIS_PROMPT.format(task_description=task_desc)
messages = [{
    "role": "user",
    "content": [{"type": "image", "image": f} for f in frames] + [{"type": "text", "text": prompt}],
}]
raw_response = generate(messages)

print("=" * 80)
print("RAW RESPONSE:")
print("=" * 80)
print(raw_response)

In [ ]:
json_match = re.search(r"\{.*\}", raw_response, re.DOTALL)
result = json.loads(json_match.group()) if json_match else {"raw_response": raw_response}
print(json.dumps(result, indent=2))

## 5. Batch analysis — run on all failure videos

In [ ]:
all_results = []
for i, video_path in enumerate(failure_videos[:10]):
    print(f"\n[{i+1}/{min(len(failure_videos), 10)}] {video_path.parent.name}/{video_path.name}")
    result = analyze_video(video_path)
    all_results.append(result)
    score = result.get("progress_score")
    if score is not None:
        print(f"  Progress: {score} | Cause: {result.get('root_cause', '?')}")
        print(f"  Fix: {result.get('corrective_suggestion', '?')[:100]}")
    else:
        print(f"  Response: {result.get('raw_response', '')[:200]}")

print(f"\nAnalyzed {len(all_results)} videos")

In [ ]:
output_path = Path("gemma4/analysis_results.json")
output_path.write_text(json.dumps(all_results, indent=2))
print(f"Saved to {output_path}")

## 6. Summary — Is Gemma 4 viable as a reward judge?

In [ ]:
# Quick summary of results
parsed_results = [r for r in all_results if "progress_score" in r]
if parsed_results:
    scores = [r["progress_score"] for r in parsed_results]
    root_causes = [r.get("root_cause", "unknown") for r in parsed_results]

    print(f"Parsed {len(parsed_results)}/{len(all_results)} responses as JSON")
    print(f"Mean progress score: {np.mean(scores):.2f}")
    print(f"Score range: [{min(scores):.2f}, {max(scores):.2f}]")
    print(f"\nRoot cause distribution:")
    for cause in set(root_causes):
        print(f"  {cause}: {root_causes.count(cause)}")
else:
    print("No results were parsed as JSON — check raw responses above")